# Ronda 4: detector especialista en filamentos pequeños

La ronda 3 mostró que el detector general (Sol entero a 1024 px) ya tocó techo: ve el 90 % de los filamentos
grandes, pero 467 de los 547 que se le escapan miden menos de 3000 px.

1. **Detector general**: el de la ronda 3, sin volver a entrenarlo (se cargan sus pesos).
2. **Detector especialista** (nuevo): mismo Mask R-CNN, pero mira la imagen original de **2048 px** por ventanas
   de 1024 (cada filamento se ve el doble de grande). Arranca de lo que aprendió el general y estudia con
   ventanas centradas casi siempre en filamentos pequeños, hasta la meseta como en la ronda 3.
3. **Dos jueces**: el del general (como en la ronda 3) y el del especialista, que solo deja pasar filamentos de
   menos de 3000 px que el general no haya encontrado ya (si se repiten, gana el general).
4. **Refinador**: el de la ronda 3, sin volver a entrenarlo, aplicado a todo lo que pasa los jueces.

Se mide por etapas en el mismo examen de 121 imágenes: general solo, general + especialista, y con refinador.
Genera `submission.csv` pero **no envía nada a la competición.**

Requisitos en Kaggle: *Accelerator* = GPU T4, *Internet* = activado, y como *Input*: la competición y el
*Output* del notebook `filamentos-detector-ronda-2` (versión de la ronda 3, con `detector_mejor.pt` y
`refinador.pt`).

In [ ]:
# Configuración. PRUEBA_RAPIDA=True reduce todo para comprobar que el notebook funciona.
PRUEBA_RAPIDA = False

RES = 1024            # resolución del detector general (la original es 2048)
# Especialista: estudia hasta que la nota se aplana (meseta), igual que el detector en la ronda 3
EPOCAS_MAX = 40       # tope de seguridad
LOTE_ESP = 2
LR_ESP = 0.005
P_PEQUENO = 0.8       # 8 de cada 10 ventanas de estudio se centran en un filamento pequeño
EXAMEN_CADA = 2
PACIENCIA = 3
MEJORA_MIN = 0.005
ACHIQUES = 1
HORAS_MAX_ESP = 4.5   # el general y el refinador ya vienen entrenados de la ronda 3
SEMILLA = 42

if PRUEBA_RAPIDA:
    EPOCAS_MAX, EXAMEN_CADA, PACIENCIA, MEJORA_MIN = 2, 1, 1, 1.0

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=True)
    import segmentation_models_pytorch as smp
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

## Código del proyecto
Copiado del repositorio `filamentos-solares-2026`.

In [ ]:
import os
os.makedirs('filamentos', exist_ok=True)

In [ ]:
%%writefile filamentos/__init__.py
"""Código del proyecto de segmentación de filamentos solares."""

In [ ]:
%%writefile filamentos/datos.py
"""Lectura de MAGFiLO y conversión entre polígonos, máscaras y RLE."""
import json
import os
from collections import defaultdict

import numpy as np
from pycocotools import mask as mask_utils

TAM = 2048


def cargar_anotaciones(raiz):
    """Devuelve {nombre_imagen: {anotador: [anotación COCO, ...]}}.

    Una imagen puede tener hasta 3 registros, uno por anotador. El id del registro
    es "GGGGAA-<nombre>"; usamos GGGGAA como identificador del anotador.
    """
    with open(os.path.join(raiz, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")) as f:
        coco = json.load(f)
    registro = {i["id"]: (i["file_name"], i["id"].split("-")[0]) for i in coco["images"]}
    out = defaultdict(lambda: defaultdict(list))
    for reg_id, (nombre, anot) in registro.items():
        out[nombre][anot]  # imágenes anotadas sin filamentos también cuentan
    for a in coco["annotations"]:
        nombre, anot = registro[a["image_id"]]
        out[nombre][anot].append(a)
    return {k: dict(v) for k, v in out.items()}


def poligono_a_mascara(segmentacion, alto=TAM, ancho=TAM):
    rles = mask_utils.frPyObjects(segmentacion, alto, ancho)
    return mask_utils.decode(mask_utils.merge(rles)).astype(bool)


def mascaras_de(anotaciones):
    return [poligono_a_mascara(a["segmentation"]) for a in anotaciones]


def mascara_a_rle(mascara):
    """Cadena de 'counts' RLE de COCO, el formato de la columna segmentation_rle."""
    rle = mask_utils.encode(np.asfortranarray(mascara.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def mascara_a_poligonos(mascara, tolerancia=1.0):
    """Contornos de una máscara como listas [x0, y0, x1, y1, ...] (formato COCO), para el visor."""
    from scipy import ndimage as ndi
    from skimage import measure

    ys, xs = ndi.find_objects(mascara.astype(np.uint8))[0]
    local = np.pad(mascara[ys, xs], 1)
    poligonos = []
    for c in measure.find_contours(local.astype(np.uint8), 0.5):
        c = measure.approximate_polygon(c, tolerancia)
        if len(c) >= 3:
            poligonos.append([round(float(v), 1) for y, x in c for v in (x + xs.start - 1, y + ys.start - 1)])
    return poligonos


def rle_a_mascara(counts, alto=TAM, ancho=TAM):
    return mask_utils.decode({"size": [alto, ancho], "counts": counts.encode("ascii")}).astype(bool)

In [ ]:
%%writefile filamentos/metrica.py
"""Panoptic Quality (PQ) tal como la define la competición, más diagnósticos.

Un filamento predicho y uno real se emparejan si su IoU > 0.5. Con ese umbral
cada filamento solo puede emparejarse con uno del otro lado, así que no hace
falta un emparejamiento óptimo.

    PQ = (suma de IoU de los aciertos / TP) * (TP / (TP + FP/2 + FN/2))
"""
from dataclasses import dataclass, field

import numpy as np

UMBRAL_IOU = 0.5


@dataclass
class Resultado:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    suma_iou: float = 0.0
    # Diagnósticos que la rúbrica también mira
    uno_a_muchos: int = 0   # un filamento real cubierto por 2+ predichos (fragmentación)
    muchos_a_uno: int = 0   # un predicho que cubre 2+ reales (fusión)
    ious: list = field(default_factory=list)

    @property
    def sq(self):
        return self.suma_iou / self.tp if self.tp else 0.0

    @property
    def rq(self):
        den = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / den if den else 1.0

    @property
    def pq(self):
        return self.sq * self.rq

    def __iadd__(self, otro):
        for k in ("tp", "fp", "fn", "suma_iou", "uno_a_muchos", "muchos_a_uno"):
            setattr(self, k, getattr(self, k) + getattr(otro, k))
        self.ious += otro.ious
        return self


def _matriz_solape(reales, predichas):
    """Intersecciones y áreas usando etiquetas, sin comparar máscara contra máscara."""
    if not reales or not predichas:
        return np.zeros((len(reales), len(predichas))), None, None
    forma = reales[0].shape
    area_r = np.array([m.sum() for m in reales], float)
    area_p = np.array([m.sum() for m in predichas], float)
    inter = np.zeros((len(reales), len(predichas)))
    for i, r in enumerate(reales):
        # Recorte a la caja de r para no recorrer la imagen entera
        ys, xs = np.nonzero(r)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        rr = r[y0:y1, x0:x1]
        for j, p in enumerate(predichas):
            inter[i, j] = np.logical_and(rr, p[y0:y1, x0:x1]).sum()
    return inter, area_r, area_p


def evaluar_imagen(reales, predichas):
    """reales, predichas: listas de máscaras booleanas 2D del mismo tamaño."""
    res = Resultado()
    inter, area_r, area_p = _matriz_solape(reales, predichas)
    if area_r is None:
        res.fn, res.fp = len(reales), len(predichas)
        return res
    iou = inter / (area_r[:, None] + area_p[None, :] - inter)
    emparejados_r = set()
    emparejados_p = set()
    for i, j in zip(*np.nonzero(iou > UMBRAL_IOU)):
        emparejados_r.add(i)
        emparejados_p.add(j)
        res.suma_iou += iou[i, j]
        res.ious.append(float(iou[i, j]))
    res.tp = len(emparejados_r)
    res.fn = len(reales) - res.tp
    res.fp = len(predichas) - len(emparejados_p)

    # Un solape cuenta si cubre al menos el 10 % del filamento más pequeño de la pareja
    toca = inter > 0.1 * np.minimum(area_r[:, None], area_p[None, :])
    res.uno_a_muchos = int((toca.sum(axis=1) >= 2).sum())
    res.muchos_a_uno = int((toca.sum(axis=0) >= 2).sum())
    return res


def resumen(res):
    return (f"PQ {res.pq:.4f} (forma SQ {res.sq:.3f} x detección RQ {res.rq:.3f}) | "
            f"TP {res.tp} FP {res.fp} FN {res.fn} | "
            f"fragmentados {res.uno_a_muchos} fusionados {res.muchos_a_uno}")


def detalle_errores(reales, predichas):
    """Para cada filamento real y predicho: si acertó y su mejor IoU con el otro lado.

    Sirve para separar los errores "de verdad" (IoU casi 0: inventado o no visto) de los
    "casi aciertos" (IoU 0,1-0,5: lo vio, pero lo partió, lo fusionó o dibujó mal el borde).
    """
    inter, area_r, area_p = _matriz_solape(reales, predichas)
    if area_r is None:
        return [(False, 0.0)] * len(reales), [(False, 0.0)] * len(predichas)
    iou = inter / (area_r[:, None] + area_p[None, :] - inter)
    mejor_r, mejor_p = iou.max(axis=1), iou.max(axis=0)
    return ([(bool(v > UMBRAL_IOU), float(v)) for v in mejor_r],
            [(bool(v > UMBRAL_IOU), float(v)) for v in mejor_p])

In [ ]:
%%writefile filamentos/clasico.py
"""Línea base clásica: observador (zonas oscuras) + juez (reglas observadas).

Sin aprendizaje automático. Sirve para estrenar el formato de envío y como
referencia contra la que medir los modelos.
"""
from dataclasses import dataclass

import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage import measure, morphology


@dataclass
class Parametros:
    umbral_oscuro: float = 0.88    # intensidad relativa al fondo local
    radio_max: float = 0.90        # regla 2: nada más allá del 90 % del radio
    area_min: int = 330            # regla 3: tamaño mínimo
    oscuridad_min: float = 0.44    # regla 4: lo más negro son manchas solares
    alargamiento_min: float = 1.8  # regla 5: forma alargada
    union_px: int = 6              # trozos a menos de esta distancia se unen en un filamento


def disco_solar(img):
    """Centro y radio del disco a partir de los píxeles brillantes."""
    m = img > 0.3 * np.percentile(img, 99)
    m = ndi.binary_fill_holes(ndi.binary_opening(m, iterations=3))
    lab, _ = ndi.label(m)
    k = np.argmax(np.bincount(lab.ravel())[1:]) + 1
    yy, xx = np.nonzero(lab == k)
    return xx.mean(), yy.mean(), np.sqrt(len(xx) / np.pi)


def normalizar(img, cx, cy, R):
    """Divide por el fondo local (mediana en ventanas de ~100 px)."""
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    peque = np.asarray(Image.fromarray(relleno.astype(np.float32)).resize((512, 512), Image.BILINEAR))
    fondo = ndi.median_filter(peque, size=25)
    fondo = np.asarray(Image.fromarray(fondo).resize(img.shape[::-1], Image.BILINEAR))
    norm = img / np.maximum(fondo, 1)
    norm[~dentro] = 1
    return norm, r


def observador(norm, r, p):
    """Candidatos: zonas oscuras dentro del disco, uniendo trozos muy cercanos."""
    oscuro = (norm < p.umbral_oscuro) & (r < p.radio_max)
    oscuro = ndi.binary_opening(oscuro, iterations=1)
    # Los trozos cercanos se agrupan con una dilatación, pero la máscara final es la original
    grupos = measure.label(ndi.binary_dilation(oscuro, structure=morphology.disk(p.union_px)))
    return grupos * oscuro


def juez(etiquetas, norm, p):
    """Aplica las reglas a cada candidato y devuelve las máscaras aceptadas."""
    aceptadas = []
    for reg in measure.regionprops(etiquetas):
        if reg.area < p.area_min:
            continue
        y0, x0, y1, x1 = reg.bbox
        vals = norm[y0:y1, x0:x1][reg.image]
        if vals.min() < p.oscuridad_min:
            continue
        if reg.axis_major_length / max(reg.axis_minor_length, 1) < p.alargamiento_min:
            continue
        m = np.zeros(etiquetas.shape, bool)
        m[y0:y1, x0:x1] = reg.image
        aceptadas.append(m)
    return aceptadas


def predecir(ruta_imagen, p=Parametros()):
    img = np.asarray(Image.open(ruta_imagen).convert("L"), float)
    cx, cy, R = disco_solar(img)
    norm, r = normalizar(img, cx, cy, R)
    return juez(observador(norm, r, p), norm, p)

In [ ]:
%%writefile filamentos/observador.py
"""Observador con red neuronal: U-Net que predice, píxel a píxel, la probabilidad de filamento.

Entrada de 3 canales a resolución reducida:
  0. imagen original normalizada a [0, 1]
  1. imagen dividida por su fondo local (resalta lo oscuro respecto al entorno)
  2. distancia al centro del disco en radios (ayuda con la regla del borde)
Objetivo: media de las máscaras de todos los anotadores (0, 1/3, 1/2, 2/3 o 1).
"""
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

from . import clasico


def preparar_imagen(ruta, res):
    """Devuelve (entrada [3,res,res] float32, r [res,res])."""
    img = np.asarray(Image.open(ruta).convert("L").resize((res, res), Image.BILINEAR), np.float32)
    cx, cy, R = clasico.disco_solar(img)
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    # Fondo local: mediana calculada a 1/4 de resolución y reampliada (≈200 veces más rápido
    # que a resolución completa; el resultado cambia menos de un 1 % de media)
    k = 4
    peq = np.asarray(Image.fromarray(relleno).resize((res // k, res // k), Image.BILINEAR))
    fondo = ndi.median_filter(peq, size=max(3, res // 20 // k))
    fondo = np.asarray(Image.fromarray(fondo).resize((res, res), Image.BILINEAR))
    norm = np.clip(img / np.maximum(fondo, 1), 0, 1.5)
    norm[~dentro] = 1
    entrada = np.stack([img / 255.0, norm / 1.5, np.clip(r, 0, 1.2) / 1.2]).astype(np.float32)
    return entrada, r.astype(np.float32)


def objetivo_suave(anotadores, res, tam=2048):
    """Media de las máscaras de cada anotador, dibujadas directamente a resolución res."""
    escala = res / tam
    acum = np.zeros((res, res), np.float32)
    for anns in anotadores.values():
        lienzo = Image.new("L", (res, res), 0)
        d = ImageDraw.Draw(lienzo)
        for a in anns:
            for p in a["segmentation"]:
                if len(p) >= 6:
                    d.polygon([(p[i] * escala, p[i + 1] * escala) for i in range(0, len(p), 2)], fill=1)
        acum += np.asarray(lienzo, np.float32)
    return acum / max(len(anotadores), 1)


class DatosEntreno(torch.utils.data.Dataset):
    def __init__(self, entradas, objetivos, aumentar):
        self.x, self.y, self.aumentar = entradas, objetivos, aumentar

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x, y = self.x[i], self.y[i][None]
        if self.aumentar:
            k = np.random.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (1, 2))
            if np.random.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, :, ::-1]
            x = x.copy()
            x[0] = np.clip(x[0] * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))


def perdida(logits, y):
    """BCE + Dice suave: BCE aprende píxel a píxel, Dice compensa que el filamento ocupa poco."""
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    inter = (p * y).sum((2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((2, 3)) + y.sum((2, 3)) + 1)
    return bce + dice.mean()


@torch.no_grad()
def predecir_prob(modelo, entrada, dispositivo, tta=True):
    """Probabilidad [res,res]; con TTA promedia las 8 rotaciones/volteos."""
    x = torch.from_numpy(entrada)[None].to(dispositivo)
    vistas = [(k, f) for k in range(4) for f in (False, True)] if tta else [(0, False)]
    acum = 0
    for k, f in vistas:
        xi = torch.rot90(x, k, (2, 3))
        if f:
            xi = xi.flip(3)
        p = torch.sigmoid(modelo(xi))
        if f:
            p = p.flip(3)
        acum = acum + torch.rot90(p, -k, (2, 3))
    return (acum / len(vistas))[0, 0].float().cpu().numpy()


def a_instancias(prob, r, umbral=0.5, area_min=330, area_max=60000, radio_max=0.95, union_px=0, tam=2048):
    """Probabilidad a resolución reducida -> lista de máscaras booleanas a 2048x2048."""
    prob_g = np.asarray(Image.fromarray(prob).resize((tam, tam), Image.BILINEAR))
    r_g = np.asarray(Image.fromarray(r).resize((tam, tam), Image.BILINEAR))
    # Solo dentro del disco: fuera no hay filamentos y una red poco entrenada puede dar ruido
    fil = (prob_g > umbral) & (r_g < 1.0)
    if union_px:
        grupos, n = ndi.label(ndi.binary_dilation(fil, iterations=union_px))
        etiquetas = grupos * fil
    else:
        etiquetas, n = ndi.label(fil, structure=np.ones((3, 3)))
    mascaras = []
    for k, sl in enumerate(ndi.find_objects(etiquetas), start=1):
        if sl is None:
            continue
        m_local = etiquetas[sl] == k
        area = m_local.sum()
        # El filamento anotado más grande mide ~38 000 px; algo mucho mayor es un error
        if area < area_min or area > area_max:
            continue
        if r_g[sl][m_local].mean() > radio_max:
            continue
        m = np.zeros((tam, tam), bool)
        m[sl] = m_local
        mascaras.append(m)
    return mascaras

In [ ]:
%%writefile filamentos/detector.py
"""Detector + refinador: cada filamento como un objeto separado.

1. Detector (Mask R-CNN, preentrenado en COCO): a 1024 px propone cada filamento con
   una caja, una confianza y una silueta. Partir y unir lo decide él, no una regla.
2. Refinador (U-Net pequeña): recorta cada filamento detectado en la imagen original
   de 2048 px, con margen alrededor, y redibuja su borde a 256x256.

Arquitectura inspirada en el notebook público de Nomannic19 ("Mask R-CNN + U-Net
Refiner"); el código es nuestro.
"""
import numpy as np
import torch
from PIL import Image
from pycocotools import mask as mask_utils
from scipy import ndimage as ndi
from scipy.sparse.csgraph import connected_components

TAM = 2048
RECORTE = 256


# ---------------------------------------------------------------- detector

def construir_detector(preentrenado=True, res=1024):
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
    from torchvision.models.detection.anchor_utils import AnchorGenerator
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    from torchvision.ops import MultiScaleRoIAlign

    pesos = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if preentrenado else None
    m = maskrcnn_resnet50_fpn_v2(weights=pesos, weights_backbone=None)
    # Dos clases: fondo y filamento
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(
        m.roi_heads.mask_predictor.conv5_mask.in_channels, m.roi_heads.mask_predictor.conv5_mask.out_channels, 2)
    # Cajas desde 16 px y alargadas (los filamentos miden 2-6 veces más de largo que de ancho)
    m.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)),
                                             aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    # Siluetas a 28x28 en vez de 14x14: más detalle en filamentos finos
    m.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)
    m.roi_heads.detections_per_img = 100
    m.transform.min_size, m.transform.max_size = (res,), res
    return m


def cargar_gris(ruta, res):
    return np.asarray(Image.open(ruta).convert("L").resize((res, res), Image.BILINEAR), np.uint8)


def mascaras_res(anotaciones, res, tam=TAM):
    """Una máscara booleana [res,res] por filamento de un anotador.

    Se dibuja a 2048 con pycocotools (como la métrica oficial) y se reduce promediando.
    Dibujar el polígono directamente a baja resolución con PIL engorda los bordes: con
    máscaras perfectas, la PQ caía de 0,89 a 0,82 en una prueba con 15 imágenes."""
    out = []
    for a in anotaciones:
        rles = mask_utils.frPyObjects(a["segmentation"], tam, tam)
        m = mask_utils.decode(mask_utils.merge(rles)).astype(np.float32)
        m = np.asarray(Image.fromarray(m).resize((res, res), Image.BILINEAR)) > 0.5
        if m.any():
            out.append(m)
    return out


def _fotometria(img):
    """Cambios de brillo, contraste y gamma solo dentro del disco (fuera es negro)."""
    dentro = img > 0.01
    if np.random.rand() < 0.7:
        img[dentro] = img[dentro] ** np.random.uniform(0.8, 1.25)
    if np.random.rand() < 0.7:
        media = img[dentro].mean()
        img[dentro] = ((img[dentro] - media) * np.random.uniform(0.85, 1.15) + media) * np.random.uniform(0.9, 1.1)
    return np.clip(img, 0, 1)


class DatosDetector(torch.utils.data.Dataset):
    """Un ejemplo por (imagen, anotador): el detector aprende con siluetas nítidas de un experto,
    no con la media borrosa. Las versiones de una misma imagen comparten la imagen en memoria y
    las máscaras se dibujan al vuelo desde los polígonos (guardarlas todas no cabe en RAM)."""

    def __init__(self, ejemplos, res, aumentar):
        self.ejemplos, self.res, self.aumentar = ejemplos, res, aumentar  # [(imagen uint8 [res,res], anotaciones)]

    def __len__(self):
        return len(self.ejemplos)

    def __getitem__(self, i):
        img, anotaciones = self.ejemplos[i]
        return objetivo_aumentado(img, mascaras_res(anotaciones, self.res), self.aumentar)


def objetivo_aumentado(img, mascaras, aumentar):
    """img uint8 [H,W] + máscaras booleanas -> (tensor imagen, objetivo de Mask R-CNN), con giros y
    cambios de brillo si aumentar. Lo comparten el detector general y el especialista."""
    img = img.astype(np.float32) / 255
    m = np.stack(mascaras) if mascaras else np.zeros((0,) + img.shape, bool)
    if aumentar:
        k = np.random.randint(4)
        img, m = np.rot90(img, k), np.rot90(m, k, (1, 2))
        if np.random.rand() < 0.5:
            img, m = img[:, ::-1], m[:, :, ::-1]
        img = _fotometria(img.copy())
    img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
    cajas = np.zeros((len(m), 4), np.float32)
    for j, mj in enumerate(m):
        ys, xs = np.nonzero(mj)
        cajas[j] = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
    objetivo = {"boxes": torch.from_numpy(cajas), "labels": torch.ones(len(m), dtype=torch.int64),
                "masks": torch.from_numpy(m.astype(np.uint8))}
    return torch.from_numpy(img)[None].repeat(3, 1, 1), objetivo


def juntar(lote):
    return tuple(zip(*lote))


@torch.no_grad()
def detectar(modelo, img, disp, umbral_mascara=0.5):
    """img uint8 [res,res] -> (máscaras booleanas [N,res,res], confianzas [N]) sin filtrar por confianza."""
    modelo.eval()
    x = torch.from_numpy(img.astype(np.float32) / 255)[None].repeat(3, 1, 1).to(disp)
    s = modelo([x])[0]
    m = (s["masks"][:, 0] >= umbral_mascara).cpu().numpy()
    conf = s["scores"].float().cpu().numpy()
    activas = m.any(axis=(1, 2))
    return m[activas], conf[activas]


def a_rle(m):
    return mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))


def grupos_solapados(mascaras):
    """Índices agrupados: dos detecciones que se tocan (IoU > 0) van al mismo grupo."""
    n = len(mascaras)
    if n < 2:
        return [[i] for i in range(n)]
    rles = [a_rle(m) for m in mascaras]
    toca = np.asarray(mask_utils.iou(rles, rles, [0] * n)) > 0
    _, etiqueta = connected_components(toca, directed=False)
    return [list(np.flatnonzero(etiqueta == k)) for k in range(etiqueta.max() + 1)]


def a_2048(m):
    """Amplía con suavizado bilineal: el borde queda liso en vez de en escalones de 2 px."""
    return np.asarray(Image.fromarray(m.astype(np.float32)).resize((TAM, TAM), Image.BILINEAR)) > 0.5


def brillo_relativo(img, r):
    """Brillo de cada píxel dividido por el fondo local (mediana en ventanas de ~100 px a 2048).

    Mismo cálculo que el análisis de manchas (reglas 4 y 5): corrige el oscurecimiento hacia el
    borde y la iluminación desigual. 1 = como el fondo; 0,5 = la mitad de brillante.
    """
    img = img.astype(np.float32)
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    peque = np.asarray(Image.fromarray(relleno).resize((512, 512), Image.BILINEAR))
    fondo = ndi.median_filter(peque, size=25)
    fondo = np.asarray(Image.fromarray(fondo).resize(img.shape[::-1], Image.BILINEAR))
    rel = img / np.maximum(fondo, 1)
    rel[~dentro] = 1
    return rel.astype(np.float16)


def alargamiento(m):
    """Eje mayor / eje menor de la elipse equivalente (como skimage regionprops)."""
    ys, xs = np.nonzero(m)
    if len(xs) < 3:
        return 1.0
    l = np.linalg.eigvalsh(np.cov(np.stack([xs, ys]).astype(np.float64)))
    return float(np.sqrt(max(l[1], 1e-9) / max(l[0], 1e-9)))


def instancias(mascaras, conf, r, umbral=0.6, fusionar=True, area_min=0, radio_max=0.95,
               rel=None, oscuro_min=None, alarg_max=None):
    """Detecciones a [res,res] -> lista de máscaras a 2048 tras el juez.

    fusionar: las detecciones que se solapan se unen en un solo filamento (Mask R-CNN
    a veces propone el mismo filamento dos veces, o dos trozos de él).
    r: distancia al centro en radios a resolución res.
    oscuro_min (regla 4): se descarta si su punto más oscuro baja de este brillo relativo
        (rel = brillo_relativo); lo negrísimo suele ser una mancha solar.
    alarg_max (regla 5): se descarta si es más alargado que esto (hilos larguísimos).
    """
    keep = conf >= umbral
    mascaras = mascaras[keep]
    if fusionar:
        mascaras = [np.logical_or.reduce(mascaras[g]) for g in grupos_solapados(mascaras)]
    else:
        # Sin fusionar, un píxel solo puede pertenecer a un filamento: gana el de más confianza
        ocupado = np.zeros(r.shape, bool)
        limpias = []
        for m in mascaras:
            m = m & ~ocupado
            ocupado |= m
            if m.any():
                limpias.append(m)
        mascaras = limpias
    out = []
    escala2 = (TAM / r.shape[0]) ** 2
    for m in mascaras:
        if m.sum() * escala2 < area_min or r[m].mean() > radio_max:
            continue
        if oscuro_min is not None and float(rel[m].min()) < oscuro_min:
            continue
        if alarg_max is not None and alargamiento(m) > alarg_max:
            continue
        out.append(a_2048(m))
    return out


# ---------------------------------------------------------------- refinador

def caja_cuadrada(m, margen=1.8, minimo=96, tam=TAM):
    """Cuadrado centrado en el filamento, margen veces su lado mayor (contexto alrededor)."""
    ys, xs = np.nonzero(m)
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    lado = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * margen))
    lado = min(max(lado, minimo), tam)
    x0, y0 = int(round(cx - lado / 2)), int(round(cy - lado / 2))
    x0, y0 = min(max(x0, 0), tam - lado), min(max(y0, 0), tam - lado)
    return x0, y0, x0 + lado, y0 + lado


def recortar(img, caja, interp=Image.BILINEAR):
    x0, y0, x1, y1 = caja
    return np.asarray(Image.fromarray(img[y0:y1, x0:x1]).resize((RECORTE, RECORTE), interp))


def recortes_entreno(img2048, mascaras2048, aumentar):
    """Pares (recorte de imagen, recorte de máscara) a 256x256, uno por filamento.

    Con aumentar=True el margen y el centro varían un poco, para que el refinador no dependa
    de que el detector encuadre el filamento perfectamente."""
    out = []
    for m in mascaras2048:
        margen = np.random.uniform(1.5, 2.2) if aumentar else 1.8
        caja = caja_cuadrada(m, margen)
        if aumentar:
            lado = caja[2] - caja[0]
            dx, dy = np.random.randint(-lado // 10, lado // 10 + 1, 2)
            x0, y0 = min(max(caja[0] + dx, 0), TAM - lado), min(max(caja[1] + dy, 0), TAM - lado)
            caja = (x0, y0, x0 + lado, y0 + lado)
        out.append((recortar(img2048, caja), recortar(m.astype(np.uint8), caja, Image.NEAREST)))
    return out


class DatosRecortes(torch.utils.data.Dataset):
    def __init__(self, imagenes, mascaras):
        self.x, self.y = imagenes, mascaras  # uint8 [N,256,256]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x, y = self.x[i].astype(np.float32) / 255, self.y[i].astype(np.float32)
        k = np.random.randint(4)
        x, y = np.rot90(x, k), np.rot90(y, k)
        if np.random.rand() < 0.5:
            x, y = x[:, ::-1], y[:, ::-1]
        x = np.clip(x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.04, 0.04), 0, 1)
        return (torch.from_numpy(np.ascontiguousarray(x))[None].repeat(3, 1, 1),
                torch.from_numpy(np.ascontiguousarray(y))[None])


@torch.no_grad()
def refinar(refinador, img2048, mascaras2048, disp, umbral=0.85, lote=32):
    """Redibuja cada filamento. Si el refinador no ve nada, se queda la silueta del detector."""
    if not mascaras2048:
        return []
    refinador.eval()
    cajas = [caja_cuadrada(m) for m in mascaras2048]
    recs = np.stack([recortar(img2048, c) for c in cajas]).astype(np.float32) / 255
    probs = []
    for i in range(0, len(recs), lote):
        x = torch.from_numpy(recs[i:i + lote])[:, None].repeat(1, 3, 1, 1).to(disp)
        # Promedio de 4 vistas volteadas (TTA)
        p = 0
        for dims in ((), (3,), (2,), (2, 3)):
            xi = x.flip(dims) if dims else x
            pi = torch.sigmoid(refinador(xi).float())
            p = p + (pi.flip(dims) if dims else pi)
        probs.append((p / 4)[:, 0].cpu().numpy())
    probs = np.concatenate(probs)
    out = []
    for m, (x0, y0, x1, y1), p in zip(mascaras2048, cajas, probs):
        nueva = np.asarray(Image.fromarray(p).resize((x1 - x0, y1 - y0), Image.BILINEAR)) >= umbral
        if not nueva.any():
            out.append(m)
            continue
        # Nos quedamos con la parte conectada que toca la silueta del detector: el recorte puede
        # incluir trozos de filamentos vecinos que no son este
        etiquetas, _ = ndi.label(nueva)
        ids = np.unique(etiquetas[m[y0:y1, x0:x1] & (etiquetas > 0)])
        if not len(ids):
            out.append(m)
            continue
        nueva = np.isin(etiquetas, ids)
        completa = np.zeros((TAM, TAM), bool)
        completa[y0:y1, x0:x1] = nueva
        out.append(completa)
    # Si dos filamentos refinados se pisan, se unen (igual que en el detector)
    return [np.logical_or.reduce([out[i] for i in g]) for g in grupos_solapados(out)]

In [ ]:
%%writefile filamentos/especialista.py
"""Detector especialista en filamentos pequeños (ronda 4).

El detector general mira el Sol entero reducido a 1024 px. El especialista mira la imagen
original de 2048 px por trozos ("ventanas") de 1024, así que cada filamento se ve el doble
de grande. Es el mismo Mask R-CNN (detector.construir_detector), arrancando de lo que ya
aprendió el general.

- Estudio: cada vez que ve un ejemplo (imagen x experto) recorta una ventana al azar; casi
  siempre centrada en un filamento pequeño, que es donde falla el general. Ve todos los
  filamentos de la ventana, también trozos de los grandes, para no aprender que eso es fondo.
- Examen y test: 9 ventanas solapadas (pasos de 512). Cada detección se queda solo en la
  ventana que contiene su centro en su zona propia, para no contarla dos veces.
- Juez del especialista: solo deja pasar filamentos de menos de AREA_MAX px (a 2048) y
  descarta lo que ya encontró el detector general (gana el general).
"""
import numpy as np
import torch
from pycocotools import mask as mask_utils
from scipy.sparse.csgraph import connected_components

from . import detector as D

TAM = D.TAM
VENTANA = 1024
PASO = 512
AREA_MAX = 3000


def posiciones(tam=TAM, ventana=VENTANA, paso=PASO):
    return list(range(0, tam - ventana + 1, paso))


def zona_propia(i, tam=TAM, ventana=VENTANA, paso=PASO):
    """[a, b) de la ventana i en un eje: la parte que le toca a ella y no a la vecina."""
    ps = posiciones(tam, ventana, paso)
    borde = (ventana - paso) // 2
    a = 0 if i == 0 else ps[i] + borde
    b = tam if i == len(ps) - 1 else ps[i] + ventana - borde
    return a, b


# ---------------------------------------------------------------- estudio

def _rles_2048(anotaciones):
    return [mask_utils.merge(mask_utils.frPyObjects(a["segmentation"], TAM, TAM)) for a in anotaciones]


class DatosEspecialista(torch.utils.data.Dataset):
    """Un ejemplo por (imagen a 2048, anotador); en cada vuelta una ventana distinta.

    p_pequeno: probabilidad de centrar la ventana en un filamento de menos de AREA_MAX px
    (con un desplazamiento al azar); si no, ventana al azar sobre el disco."""

    def __init__(self, ejemplos, aumentar, p_pequeno=0.8):
        self.ejemplos, self.aumentar, self.p_pequeno = ejemplos, aumentar, p_pequeno  # [(img uint8 2048, anotaciones)]

    def __len__(self):
        return len(self.ejemplos)

    def __getitem__(self, i):
        img, anotaciones = self.ejemplos[i]
        rles = _rles_2048(anotaciones)
        areas = mask_utils.area(rles) if rles else np.zeros(0)
        cajas = mask_utils.toBbox(rles) if rles else np.zeros((0, 4))  # x, y, ancho, alto
        pequenos = np.flatnonzero(areas < AREA_MAX)
        tope = TAM - VENTANA
        if len(pequenos) and np.random.rand() < self.p_pequeno:
            x, y, w, h = cajas[np.random.choice(pequenos)]
            cx, cy = x + w / 2 + np.random.uniform(-256, 256), y + h / 2 + np.random.uniform(-256, 256)
        else:
            cx, cy = np.random.uniform(VENTANA / 2, TAM - VENTANA / 2, 2)
        x0 = int(np.clip(round(cx - VENTANA / 2), 0, tope))
        y0 = int(np.clip(round(cy - VENTANA / 2), 0, tope))
        ventana = img[y0:y0 + VENTANA, x0:x0 + VENTANA]
        mascaras = []
        for r, (x, y, w, h) in zip(rles, cajas):
            if x + w <= x0 or x >= x0 + VENTANA or y + h <= y0 or y >= y0 + VENTANA:
                continue
            m = mask_utils.decode(r)[y0:y0 + VENTANA, x0:x0 + VENTANA].astype(bool)
            if m.sum() >= 20:  # trozos minúsculos del borde de la ventana no enseñan nada
                mascaras.append(m)
        return D.objetivo_aumentado(ventana, mascaras, self.aumentar)


# ---------------------------------------------------------------- examen y test

def ventanas(img2048):
    ps = posiciones()
    for iy, y0 in enumerate(ps):
        for ix, x0 in enumerate(ps):
            yield img2048[y0:y0 + VENTANA, x0:x0 + VENTANA], x0, y0, zona_propia(ix), zona_propia(iy)


@torch.no_grad()
def detectar(modelo, img2048, disp, conf_min=0.2):
    """Detecciones pequeñas en toda la imagen: (lista de RLE a 2048, confianzas)."""
    rles, confs = [], []
    for v, x0, y0, (ax, bx), (ay, by) in ventanas(img2048):
        with torch.autocast(disp, enabled=disp == "cuda"):
            m, c = D.detectar(modelo, v, disp)
        for mj, cj in zip(m, c):
            if cj < conf_min:
                continue
            ys, xs = np.nonzero(mj)
            if len(xs) >= AREA_MAX:
                continue
            cx, cy = x0 + xs.mean(), y0 + ys.mean()
            if not (ax <= cx < bx and ay <= cy < by):
                continue  # este filamento le toca a la ventana vecina
            completa = np.zeros((TAM, TAM), np.uint8)
            completa[y0:y0 + VENTANA, x0:x0 + VENTANA] = mj
            rles.append(D.a_rle(completa))
            confs.append(float(cj))
    return rles, np.asarray(confs, np.float32)


def juez(rles, conf, r2048, generales, umbral=0.6, fusionar=True, area_min=0, radio_max=0.95,
         solape_max=0.2):
    """Detecciones del especialista -> máscaras a 2048 que se añaden a las del detector general.

    generales: máscaras a 2048 que ya dio el detector general (tras su juez).
    solape_max: si más de esta fracción del filamento cae sobre uno del general, se descarta
        (gana el general); si es menos, se le quitan esos píxeles.
    """
    elegidos = [r for r, c in zip(rles, conf) if c >= umbral]  # ya vienen de mayor a menor confianza
    if not elegidos:
        return []
    # Se descomprimen de una en una: a 2048 px, cientos de máscaras juntas no caben en memoria
    if fusionar:
        toca = np.asarray(mask_utils.iou(elegidos, elegidos, [0] * len(elegidos))) > 0
        _, etiqueta = connected_components(toca, directed=False)
        mascaras = (mask_utils.decode(mask_utils.merge([elegidos[i] for i in np.flatnonzero(etiqueta == k)],
                                                       intersect=0)).astype(bool)
                    for k in range(etiqueta.max() + 1))
    else:
        mascaras = (mask_utils.decode(r).astype(bool) for r in elegidos)
    del_general = np.logical_or.reduce(generales) if len(generales) else np.zeros((TAM, TAM), bool)
    tomado = np.zeros((TAM, TAM), bool)  # píxeles ya dados a otra detección del especialista
    out = []
    for m in mascaras:
        m = m & ~tomado  # un píxel solo puede ser de un filamento: gana el de más confianza
        area = m.sum()
        if not area_min <= area < AREA_MAX or r2048[m].mean() > radio_max:
            continue
        if (m & del_general).sum() > solape_max * area:
            continue  # ya lo encontró el general
        m = m & ~del_general
        tomado |= m
        out.append(m)
    return out

## Datos, separación estudio / examen y pesos de la ronda 3
Misma separación que las rondas 1-3.

In [ ]:
import glob, hashlib, json, time, random
import numpy as np, torch
from multiprocessing import Pool
from PIL import Image
from pycocotools import mask as mask_utils
from filamentos.datos import cargar_anotaciones, mascaras_de, mascara_a_rle, rle_a_mascara
from filamentos.metrica import Resultado, evaluar_imagen, resumen
from filamentos import clasico, detector as D, especialista as E, observador as O

random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
disp = 'cuda' if torch.cuda.is_available() else 'cpu'
T0 = time.time()

def buscar(patron):
    for raiz in ('/kaggle/input', os.environ.get('MAGFILO', '.'), os.environ.get('PESOS', '.')):
        r = glob.glob(f'{raiz}/**/{patron}', recursive=True)
        if r:
            return r[0]
    return None

json_path = buscar('MAGFiLO_1.0_Annotations_kaggle2026_train.json')
assert json_path, 'No encuentro los datos: añade la competición como Input del notebook'
DATOS = os.path.dirname(os.path.dirname(json_path))
PESOS_DET, PESOS_REF = buscar('detector_mejor.pt'), buscar('refinador.pt')
if not PRUEBA_RAPIDA:
    assert PESOS_DET and PESOS_REF, ('No encuentro los pesos de la ronda 3: añade como Input el Output del '
                                     'notebook filamentos-detector-ronda-2')
print('Datos en', DATOS, '| pesos:', PESOS_DET, PESOS_REF)

ANOT = cargar_anotaciones(DATOS)
nombres = sorted(n for n in ANOT if os.path.exists(f'{DATOS}/train/train_images/{n}'))

def es_val(n):
    return int(hashlib.md5(n.encode()).hexdigest(), 16) % 100 < 15
val = [n for n in nombres if es_val(n)]
tr = [n for n in nombres if not es_val(n)]
if PRUEBA_RAPIDA:
    tr, val = tr[:8], val[:3]
print(len(tr), 'imágenes de estudio,', len(val), 'de examen')

In [ ]:
def radio(img):
    """Distancia al centro del disco solar, en radios, para cada píxel."""
    cx, cy, R = clasico.disco_solar(img.astype(np.float32))
    yy, xx = np.indices(img.shape)
    return (np.hypot(xx - cx, yy - cy) / R).astype(np.float32)

def gris2048(n, carpeta='train/train_images'):
    return np.asarray(Image.open(f'{DATOS}/{carpeta}/{n}').convert('L'), np.uint8)

def cargar(n):
    grande = gris2048(n)
    img = np.asarray(Image.fromarray(grande).resize((RES, RES), Image.BILINEAR), np.uint8)
    return grande, img, radio(img)

def radio2048(r):
    return np.asarray(Image.fromarray(r).resize((2048, 2048), Image.BILINEAR))

t = time.time()
with Pool(os.cpu_count()) as p:
    cache = dict(zip(tr + val, p.map(cargar, tr + val)))
ejemplos_esp = [(cache[n][0], anns) for n in tr for anns in ANOT[n].values()]
print(f'{len(ejemplos_esp)} ejemplos de estudio (imagen x experto), cargados en {time.time() - t:.0f} s')

# Filamentos de los expertos en el examen, a 2048 y comprimidos (RLE) para no llenar la memoria
reales_val = {n: [[D.a_rle(m) for m in mascaras_de(anns)] for anns in ANOT[n].values()] for n in val}

def pq_examen(pred_por_imagen, nombres=None):
    """pred_por_imagen: {imagen: [máscaras 2048]}. Compara contra cada experto, como en las rondas 1-3."""
    tot = Resultado()
    for n in (nombres or val):
        pred = pred_por_imagen[n]
        for rles in reales_val[n]:
            tot += evaluar_imagen([mask_utils.decode(r).astype(bool) for r in rles], pred)
    return tot

## Etapa 1: detector general de la ronda 3 y su juez
Debe dar lo mismo que en la ronda 3 (0,389 detector solo).

In [ ]:
gen = D.construir_detector(False, RES).to(disp)
if PESOS_DET:
    gen.load_state_dict(torch.load(PESOS_DET, map_location=disp))

crudas_gen = {}
for n in val:
    with torch.autocast(disp, enabled=disp == 'cuda'):
        m, c = D.detectar(gen, cache[n][1], disp)
    keep = c >= 0.2
    crudas_gen[n] = ([D.a_rle(x) for x in m[keep]], c[keep])

def juez_gen(n, **post):
    rles, conf = crudas_gen[n]
    m = np.stack([mask_utils.decode(r).astype(bool) for r in rles]) if rles else np.zeros((0, RES, RES), bool)
    return D.instancias(m, conf, cache[n][2], **post)

resultados = []
for umbral in (0.6, 0.7, 0.8, 0.85, 0.9):
    for fusionar in (True, False):
        for area_min in (0, 330):
            post = dict(umbral=umbral, fusionar=fusionar, area_min=area_min)
            res = pq_examen({n: juez_gen(n, **post) for n in val})
            resultados.append((res.pq, post))
PQ_GEN, POST_GEN = max(resultados, key=lambda t: t[0])
pred_gen = {n: juez_gen(n, **POST_GEN) for n in val}
RES_GEN = pq_examen(pred_gen)
print('Juez del general:', POST_GEN, '->', resumen(RES_GEN))

## Etapa 2: entrenamiento del especialista
Arranca con los pesos del general. En el examen se mide la nota del conjunto: general + especialista
(confianza 0,6), así la meseta se declara cuando el especialista deja de aportar.

In [ ]:
esp = D.construir_detector(False, E.VENTANA).to(disp)
if PESOS_DET:
    esp.load_state_dict(torch.load(PESOS_DET, map_location=disp))
params = [p for p in esp.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR_ESP, momentum=0.9, weight_decay=1e-4)
escalador = torch.amp.GradScaler(enabled=disp == 'cuda')
dl = torch.utils.data.DataLoader(E.DatosEspecialista(ejemplos_esp, aumentar=True, p_pequeno=P_PEQUENO),
                                 batch_size=LOTE_ESP, shuffle=True, num_workers=2, collate_fn=D.juntar)
r2048_val = {n: radio2048(cache[n][2]) for n in val}

def detectar_examen_esp(modelo):
    return {n: E.detectar(modelo, cache[n][0], disp) for n in val}

def juez_esp(crudas, n, **post):
    rles, conf = crudas[n]
    return E.juez(rles, conf, r2048_val[n], pred_gen[n], **post)

mejor, historia = -1, []
sin_mejora, achiques, t_esp = 0, 0, time.time()
for epoca in range(1, EPOCAS_MAX + 1):
    esp.train(); t = time.time(); suma = 0
    for imgs, objs in dl:
        imgs = [i.to(disp) for i in imgs]
        objs = [{k: v.to(disp) for k, v in o.items()} for o in objs]
        with torch.autocast(disp, enabled=disp == 'cuda'):
            l = sum(esp(imgs, objs).values())
        opt.zero_grad(); escalador.scale(l).backward(); escalador.step(opt); escalador.update()
        suma += l.item()
    linea = f'vuelta {epoca}: pérdida {suma / len(dl):.4f} ({time.time() - t:.0f} s, paso {opt.param_groups[0]["lr"]:g})'
    parar = False
    if epoca % EXAMEN_CADA == 0:
        crudas_esp = detectar_examen_esp(esp)
        res = pq_examen({n: pred_gen[n] + juez_esp(crudas_esp, n, umbral=0.6) for n in val})
        historia.append((epoca, round(res.pq, 4)))
        linea += ' | ' + resumen(res)
        if res.pq > mejor:
            if res.pq >= mejor + MEJORA_MIN:
                sin_mejora = 0
            else:
                sin_mejora += 1  # mejora demasiado pequeña: se guarda, pero cuenta para la meseta
            mejor = res.pq
            torch.save(esp.state_dict(), 'especialista_mejor.pt')
            linea += ' | guardado'
        else:
            sin_mejora += 1
        linea += f' | sin mejora {sin_mejora}/{PACIENCIA}'
        if sin_mejora >= PACIENCIA:
            if achiques < ACHIQUES:
                achiques += 1; sin_mejora = 0
                for g in opt.param_groups:
                    g['lr'] *= 0.1
                linea += ' | MESETA: achico el paso x0,1'
            else:
                linea += ' | MESETA otra vez: paro'
                parar = True
    if time.time() - t_esp > HORAS_MAX_ESP * 3600:
        linea += ' | tope de tiempo: paro'
        parar = True
    print(linea, flush=True)
    if parar:
        break
if not os.path.exists('especialista_mejor.pt'):
    torch.save(esp.state_dict(), 'especialista_mejor.pt')
print('Curva (vuelta, PQ general + especialista):', historia)
print(f'Entrenamiento del especialista: {epoca} vueltas en {(time.time() - t_esp) / 60:.0f} min')
print('General solo:', round(RES_GEN.pq, 4), '| mejor con especialista (confianza 0,6):', round(mejor, 4))

## Etapa 3: juez del especialista
Confianza mínima, unir o no sus detecciones que se pisan, y tamaño mínimo.

In [ ]:
esp.load_state_dict(torch.load('especialista_mejor.pt', map_location=disp))
crudas_esp = detectar_examen_esp(esp)
PQ_ESP, POST_ESP = RES_GEN.pq, None
for umbral in (0.5, 0.6, 0.7, 0.8, 0.9):
    for fusionar in (True, False):
        for area_min in (0, 330):
            post = dict(umbral=umbral, fusionar=fusionar, area_min=area_min)
            res = pq_examen({n: pred_gen[n] + juez_esp(crudas_esp, n, **post) for n in val})
            print(post, '->', resumen(res), flush=True)
            if res.pq > PQ_ESP:
                PQ_ESP, POST_ESP = res.pq, post
if POST_ESP is None:
    print('El especialista no sube la nota con ningún juez: se queda solo el general')
    extra = {n: [] for n in val}
else:
    extra = {n: juez_esp(crudas_esp, n, **POST_ESP) for n in val}
pred_det = {n: pred_gen[n] + extra[n] for n in val}
RES_DET = pq_examen(pred_det)
print('Juez del especialista:', POST_ESP)
print('General solo:        ', resumen(RES_GEN))
print('General + especialista:', resumen(RES_DET))
print(f'El especialista añade {sum(map(len, extra.values()))} filamentos: aciertos {RES_DET.tp - RES_GEN.tp:+d}, '
      f'inventados {RES_DET.fp - RES_GEN.fp:+d}, se escapan {RES_DET.fn - RES_GEN.fn:+d}')

## Etapa 4: refinador de la ronda 3
Se prueba sobre el general solo (como la ronda 3) y sobre general + especialista.

In [ ]:
ref = smp.Unet('resnet18', encoder_weights=None, in_channels=3, classes=1).to(disp)
if PESOS_REF:
    ref.load_state_dict(torch.load(PESOS_REF, map_location=disp))

def refinar_todo(pred, umbral):
    out = {}
    for n in val:
        with torch.autocast(disp, enabled=disp == 'cuda'):
            out[n] = D.refinar(ref, cache[n][0], pred[n], disp, umbral=umbral)
    return out

candidatos = [('general solo', None, pred_gen, RES_GEN), ('general + especialista', None, pred_det, RES_DET)]
for umbral_ref in (0.3, 0.4, 0.5):
    for nombre, pred in (('general solo', pred_gen), ('general + especialista', pred_det)):
        if nombre == 'general + especialista' and POST_ESP is None:
            continue
        p = refinar_todo(pred, umbral_ref)
        res = pq_examen(p)
        print(f'{nombre} + refinador (umbral {umbral_ref}):', resumen(res), flush=True)
        candidatos.append((nombre, umbral_ref, p, res))
NOMBRE_FINAL, UMBRAL_REF, pred_final, RES_FINAL = max(candidatos, key=lambda t: t[3].pq)
USAR_ESP = NOMBRE_FINAL == 'general + especialista'
print(f'Elegido: {NOMBRE_FINAL}' + (f' + refinador (umbral {UMBRAL_REF})' if UMBRAL_REF else '') + f' -> PQ {RES_FINAL.pq:.4f}')
print('Ronda 3 para comparar: 0,4405 (TP 820, FP 350, FN 547)')

## Predicciones del examen para el visor

In [ ]:
from filamentos.datos import mascara_a_poligonos
examen = {}
for n in val:
    pred = pred_final[n]
    res = pq_examen(pred_final, [n])
    examen[n] = {'pq': round(res.pq, 3), 'tp': res.tp, 'fp': res.fp, 'fn': res.fn,
                 'f': [{'a': int(m.sum()), 'p': mascara_a_poligonos(m)} for m in pred]}
with open('predicciones_examen.json', 'w') as fh:
    json.dump({'post': POST_GEN, 'especialista': POST_ESP if USAR_ESP else None, 'refinador': UMBRAL_REF,
               'imagenes': examen}, fh, separators=(',', ':'))
peores = sorted(examen.items(), key=lambda t: t[1]['pq'])[:10]
print('Las 10 imágenes del examen donde peor lo hace:')
for n, e in peores:
    print(f"  {n}: PQ {e['pq']:.2f}  (aciertos {e['tp']}, inventados {e['fp']}, se le escapan {e['fn']})")

## Informe de errores
Igual que en la ronda 3: tamaño, zona, oscuridad y telescopio; **casi** = lo vio pero lo partió, fusionó o
dibujó mal (mejor IoU 0,1-0,5); **puro** = inventado de la nada o no visto en absoluto (mejor IoU < 0,1).

In [ ]:
import pandas as pd
from filamentos.metrica import detalle_errores
TELESCOPIO = {'B': 'Big Bear', 'M': 'Mauna Loa', 'L': 'Learmonth', 'U': 'Udaipur', 'T': 'Teide', 'C': 'Cerro Tololo'}
filas = []
for n in val:
    pred = pred_final[n]
    x, r = O.preparar_imagen(f'{DATOS}/train/train_images/{n}', 1024)
    r_g = np.asarray(Image.fromarray(r).resize((2048, 2048), Image.BILINEAR))
    contraste = np.asarray(Image.fromarray(x[1]).resize((2048, 2048), Image.BILINEAR)) * 1.5
    tel = TELESCOPIO.get(os.path.splitext(n)[0][-2], '?')
    for anot, rles in zip(ANOT[n], reales_val[n]):
        reales = [mask_utils.decode(r).astype(bool) for r in rles]
        d_r, d_p = detalle_errores(reales, pred)
        for lado, mascaras, dets in (('real', reales, d_r), ('red', pred, d_p)):
            for m, (ok, iou) in zip(mascaras, dets):
                filas.append(dict(imagen=n, anotador=anot, lado=lado, acierto=ok, iou=iou, area=int(m.sum()),
                                  radio=float(r_g[m].mean()), contraste=float(contraste[m].mean()), telescopio=tel))
Err = pd.DataFrame(filas)
Err['error'] = np.where(Err.acierto, 'acierto', np.where(Err.iou >= 0.1, 'casi', 'puro'))
Err['tamaño'] = pd.cut(Err.area, [0, 330, 1000, 3000, 10000, 1e9], labels=['<330', '330-1k', '1k-3k', '3k-10k', '>10k'])
Err['zona'] = pd.cut(Err.radio, [0, 0.5, 0.8, 0.9, 2], labels=['centro <0,5', '0,5-0,8', '0,8-0,9', 'borde >0,9'])
Err['oscuridad'] = pd.cut(Err.contraste, [0, 0.8, 0.9, 0.95, 9], labels=['muy oscuro <0,8', '0,8-0,9', '0,9-0,95', 'casi fondo >0,95'])
Err.to_csv('errores_examen.csv', index=False)

def tabla(col):
    t = pd.crosstab([Err.lado, Err[col]], Err.error)
    t['% acierto'] = (100 * t.get('acierto', 0) / t.sum(axis=1)).round(0)
    return t

pd.set_option('display.width', 200)
for col in ('tamaño', 'zona', 'oscuridad', 'telescopio'):
    print(f'\n=== Por {col} (real = filamentos de los expertos; red = lo que dibujó el modelo) ===')
    print(tabla(col))

## Predicción del test y `submission.csv`
El archivo se genera pero **no se envía**.

In [ ]:
import csv
test_dir = f'{DATOS}/test/test_images'
filas = []
for f in (sorted(os.listdir(test_dir)) if os.path.isdir(test_dir) else []):
    grande = gris2048(f, 'test/test_images')
    img = np.asarray(Image.fromarray(grande).resize((RES, RES), Image.BILINEAR), np.uint8)
    rr = radio(img)
    with torch.autocast(disp, enabled=disp == 'cuda'):
        m, c = D.detectar(gen, img, disp)
    pred = D.instancias(m, c, rr, **POST_GEN)
    if USAR_ESP:
        rles, conf = E.detectar(esp, grande, disp)
        pred = pred + E.juez(rles, conf, radio2048(rr), pred, **POST_ESP)
    if UMBRAL_REF:
        with torch.autocast(disp, enabled=disp == 'cuda'):
            pred = D.refinar(ref, grande, pred, disp, umbral=UMBRAL_REF)
    img_id = os.path.splitext(f)[0]
    for k, mk in enumerate(pred, start=1):
        filas.append((f'{img_id}_{k}', mascara_a_rle(mk)))
    if PRUEBA_RAPIDA and len(filas) > 20:
        break
with open('submission.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['filament_id', 'segmentation_rle'])
    w.writerows(filas)

ids = [a for a, _ in filas]
assert len(ids) == len(set(ids))
if filas:
    assert rle_a_mascara(filas[0][1]).shape == (2048, 2048)
else:
    print('AVISO: no hay filamentos en el test (¿falta la carpeta test?)')
print(len(filas), 'filamentos en', len(set(i.rsplit('_', 1)[0] for i in ids)), 'imágenes -> submission.csv')
print(f'Tiempo total: {(time.time() - T0) / 60:.0f} min')